# Practical 6 - Text Analysis, Synonyms, Autocomplete and Partial Search

Hands-on implementation of Elasticsearch **text analysis** against the local
9.x node (`http://127.0.0.1:9200`) using the official Python client.

The six tasks are:

1. **Analyze text** - compare the standard analyzer with the other built-in
   analyzers on "Green Valley Residency in New Panvel".
2. **Custom analyzer** - build a `lowercase` + stop-word analyzer.
3. **Synonyms** - map apartment/flat, builder/developer and
   residence/residency, then prove the search behaviour changes.
4. **Autocomplete** - edge-ngram matching while the user is still typing.
5. **Partial search** - n-gram matching for any part of a project name, with a
   wildcard cross-check.
6. **Case-insensitive exact match** - a keyword normalizer for locality/city.

The notebook drops and recreates the `projects` index on every run, so it is
idempotent. Each task prints the technique, query, expected output, actual
output and explanation, and asserts the important results.

**Prerequisites:** Elasticsearch running (Practical 1), `generate_data.py`
executed in this folder, and the shared `es` kernel (see `README.md`).

## Setup - client, helpers and data

`words()` mirrors what a simple analyzer does (lowercase, split on
non-alphanumerics), which lets the notebook compute expected result sets from
`projects.json` instead of hard-coding counts.

The helper functions are:

- `recreate()` / `load()` - rebuild the index from the settings and JSON.
- `analyze_tokens()` - call the `_analyze` API with a named analyzer.
- `all_ids()` - run a query and return the matching `project_id`s.
- `name_prefix()`, `name_token_phrase()`, `name_contains()` - expected sets
  for the autocomplete and partial-search tasks.

In [1]:
import json
import re
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ES_URL = "http://127.0.0.1:9200"
INDEX = "projects"
es = Elasticsearch(ES_URL)

info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])

HERE = Path.cwd()
DATA = json.loads((HERE / "projects.json").read_text(encoding="utf-8"))
print("projects loaded:", len(DATA))


def pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def words(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def analyze_tokens(text, analyzer):
    response = es.indices.analyze(index=INDEX, analyzer=analyzer, text=text)
    return [token["token"] for token in response["tokens"]]


def recreate(index, settings, mappings):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
    es.indices.create(index=index, settings=settings, mappings=mappings)


def load(index, records):
    actions = [
        {"_index": index, "_id": record["project_id"], "_source": record}
        for record in records
    ]
    bulk(es, actions, refresh=True)


def hits_of(response):
    return response["hits"]["hits"]


def ids_of(response):
    return [hit["_id"] for hit in hits_of(response)]


def total_hits(response):
    return response["hits"]["total"]["value"]


def all_ids(query):
    response = es.search(
        index=INDEX,
        query=query,
        size=len(DATA),
        source_includes=["project_id"],
    )
    return ids_of(response)


def name_prefix(prefix):
    return {
        p["project_id"] for p in DATA
        if any(word.startswith(prefix) for word in words(p["name"]))
    }


def name_token_phrase(first, second_prefix):
    found = set()
    for project in DATA:
        tokens = words(project["name"])
        if any(
            tokens[i] == first and tokens[i + 1].startswith(second_prefix)
            for i in range(len(tokens) - 1)
        ):
            found.add(project["project_id"])
    return found


def name_contains(fragment):
    return {p["project_id"] for p in DATA if fragment in p["name"].lower()}


def desc_has(*terms):
    wanted = set(terms)
    return {p["project_id"] for p in DATA if wanted & set(words(p["description"]))}

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
projects loaded: 50


### Index settings and mappings

One index holds every analyzer so the tasks can be compared side by side:

| Element | Definition |
|---------|------------|
| `case_insensitive` normalizer | `lowercase` + `asciifolding` |
| `lower_stop` analyzer | standard tokenizer, lowercase, custom stop words |
| `synonym_analyzer` | standard tokenizer, lowercase, business synonym filter |
| `autocomplete_index` | standard tokenizer, lowercase, `edge_ngram` 2-15 |
| `autocomplete_search` | standard tokenizer, lowercase (no n-grams) |
| `ngram_index` | standard tokenizer, lowercase, `ngram` 3-4 |

`description` gets two sub-fields (`description.clean`, `description.syn`);
`name` gets `keyword`, `autocomplete` and `ngram` sub-fields; `locality` uses
the normalizer and keeps an unnormalized `locality.raw` for comparison.

In [2]:
STOPWORDS = [
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "in", "into",
    "is", "it", "near", "of", "on", "or", "the", "to", "with",
]

SYNONYM_GROUPS = [
    "apartment, apartments, flat, flats",
    "builder, builders, developer, developers",
    "residence, residences, residency, residencies",
]

INDEX_SETTINGS = {
    "analysis": {
        "normalizer": {
            "case_insensitive": {
                "type": "custom",
                "filter": ["lowercase", "asciifolding"],
            }
        },
        "filter": {
            "lower_stop_filter": {
                "type": "stop",
                "stopwords": STOPWORDS,
            },
            "business_synonyms_filter": {
                "type": "synonym",
                "synonyms": SYNONYM_GROUPS,
            },
            "edge_ngram_filter": {
                "type": "edge_ngram",
                "min_gram": 2,
                "max_gram": 15,
                "token_chars": ["letter", "digit"],
            },
            "ngram_filter": {
                "type": "ngram",
                "min_gram": 3,
                "max_gram": 4,
                "token_chars": ["letter", "digit"],
            },
        },
        "analyzer": {
            "lower_stop": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "lower_stop_filter"],
            },
            "synonym_analyzer": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "business_synonyms_filter"],
            },
            "autocomplete_index": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "asciifolding", "edge_ngram_filter"],
            },
            "autocomplete_search": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "asciifolding"],
            },
            "ngram_index": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "asciifolding", "ngram_filter"],
            },
        },
    }
}

MAPPING = {
    "properties": {
        "project_id": {"type": "keyword"},
        "name": {
            "type": "text",
            "fields": {
                "keyword": {"type": "keyword", "ignore_above": 256},
                "autocomplete": {
                    "type": "text",
                    "analyzer": "autocomplete_index",
                    "search_analyzer": "autocomplete_search",
                },
                "ngram": {"type": "text", "analyzer": "ngram_index"},
            },
        },
        "description": {
            "type": "text",
            "fields": {
                "clean": {"type": "text", "analyzer": "lower_stop"},
                "syn": {"type": "text", "analyzer": "synonym_analyzer"},
            },
        },
        "builder": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword"}},
        },
        "locality": {
            "type": "keyword",
            "normalizer": "case_insensitive",
            "fields": {"raw": {"type": "keyword"}},
        },
        "city": {
            "type": "keyword",
            "normalizer": "case_insensitive",
            "fields": {"raw": {"type": "keyword"}},
        },
        "property_type": {"type": "keyword"},
        "status": {"type": "keyword", "normalizer": "case_insensitive"},
        "price_min": {"type": "integer"},
        "price_max": {"type": "integer"},
        "bhk": {"type": "keyword"},
        "amenities": {"type": "keyword"},
        "possession_date": {"type": "date"},
        "rera_approved": {"type": "boolean"},
        "rating": {"type": "float"},
    }
}

recreate(INDEX, INDEX_SETTINGS, MAPPING)
load(INDEX, DATA)

count = es.count(index=INDEX)["count"]
print("index:", INDEX, "| documents:", count)
print("analyzers:", ", ".join(sorted(INDEX_SETTINGS["analysis"]["analyzer"])))
print("normalizer:", ", ".join(sorted(INDEX_SETTINGS["analysis"]["normalizer"])))
assert count == len(DATA) == 50

index: projects | documents: 50
analyzers: autocomplete_index, autocomplete_search, lower_stop, ngram_index, synonym_analyzer
normalizer: case_insensitive


## Task 1 - Analyze Text

`_analyze` runs the analysis chain without indexing anything. The **standard**
analyzer uses the Unicode text segmentation tokenizer and lowercases the
tokens; it keeps stop words. The others show what changes when that chain is
swapped:

- `simple` - letters only, lowercase.
- `whitespace` - split on spaces, no lowercasing.
- `stop` - standard plus English stop words (drops `in`).
- `keyword` - the whole input as one token.
- `english` - standard plus English stemming (`valley` -> `vallei`,
  `residency` -> `resid`).
- `lower_stop` - the custom analyzer built in Task 2.

In [3]:
SAMPLE = "Green Valley Residency in New Panvel"
ANALYZERS = [
    "standard", "simple", "whitespace", "stop", "keyword", "english",
    "lower_stop",
]
results = {name: analyze_tokens(SAMPLE, name) for name in ANALYZERS}

print(f"Analyze: {SAMPLE!r}")
print(f"{'analyzer':<12} {'tokens':>6}  output")
print("-" * 100)
for name in ANALYZERS:
    print(f"{name:<12} {len(results[name]):>6}  {' | '.join(results[name])}")

assert results["standard"] == [
    "green", "valley", "residency", "in", "new", "panvel"
]
assert results["keyword"] == [SAMPLE]
assert "in" in results["standard"] and "in" not in results["stop"]
assert results["whitespace"][0] == "Green"
assert "vallei" in results["english"] and "resid" in results["english"]
assert "in" not in results["lower_stop"]

print()
print("standard keeps 'in' (6 tokens); stop and lower_stop remove it (5)")
print("keyword keeps one token and the original case; whitespace keeps case")
print("english stems 'valley' and 'residency' but not 'new'/'panvel'")

Analyze: 'Green Valley Residency in New Panvel'
analyzer     tokens  output
----------------------------------------------------------------------------------------------------
standard          6  green | valley | residency | in | new | panvel
simple            6  green | valley | residency | in | new | panvel
whitespace        6  Green | Valley | Residency | in | New | Panvel
stop              5  green | valley | residency | new | panvel
keyword           1  Green Valley Residency in New Panvel
english           5  green | vallei | resid | new | panvel
lower_stop        5  green | valley | residency | new | panvel

standard keeps 'in' (6 tokens); stop and lower_stop remove it (5)
keyword keeps one token and the original case; whitespace keeps case
english stems 'valley' and 'residency' but not 'new'/'panvel'


## Task 2 - Custom Analyzer

The custom analyzer is `standard` tokenizer -> `lowercase` -> stop filter. The
stop list includes the usual English words plus real-estate filler such as
`in`, `near` and `of`, which carry no search meaning but appear in every
listing sentence.

Searching `description.clean` also proves the stop words are gone: a query that
contains only `in` analyzes to zero tokens, while the same query on the plain
`description` field matches every project.

In [4]:
print("lower_stop definition:")
print(pretty({
    "filter": {
        "lower_stop_filter": {
            "type": "stop",
            "stopwords": STOPWORDS,
        }
    },
    "analyzer": {
        "lower_stop": {
            "type": "custom",
            "tokenizer": "standard",
            "filter": ["lowercase", "lower_stop_filter"],
        }
    },
}))

custom = analyze_tokens(SAMPLE, "lower_stop")
standard = analyze_tokens(SAMPLE, "standard")
print("\ntext:", SAMPLE)
print("standard  :", " | ".join(standard))
print("lower_stop:", " | ".join(custom))

second = "Green Valley Residency near the New Panvel market"
print("\nsecond text:", second)
print("standard  :", " | ".join(analyze_tokens(second, "standard")))
print("lower_stop:", " | ".join(analyze_tokens(second, "lower_stop")))

assert custom == ["green", "valley", "residency", "new", "panvel"]
assert "near" not in analyze_tokens(second, "lower_stop")
assert "the" not in analyze_tokens(second, "lower_stop")

lower_stop definition:
{
  "filter": {
    "lower_stop_filter": {
      "type": "stop",
      "stopwords": [
        "a",
        "an",
        "and",
        "are",
        "as",
        "at",
        "be",
        "by",
        "for",
        "in",
        "into",
        "is",
        "it",
        "near",
        "of",
        "on",
        "or",
        "the",
        "to",
        "with"
      ]
    }
  },
  "analyzer": {
    "lower_stop": {
      "type": "custom",
      "tokenizer": "standard",
      "filter": [
        "lowercase",
        "lower_stop_filter"
      ]
    }
  }
}

text: Green Valley Residency in New Panvel
standard  : green | valley | residency | in | new | panvel
lower_stop: green | valley | residency | new | panvel

second text: Green Valley Residency near the New Panvel market
standard  : green | valley | residency | near | the | new | panvel | market
lower_stop: green | valley | residency | new | panvel | market


In [5]:
query = {
    "match": {
        "description.clean": {
            "query": "Residency in New Panvel",
            "operator": "and",
        }
    }
}
expected = {
    p["project_id"] for p in DATA
    if {"residency", "new", "panvel"} <= set(words(p["description"]))
}
actual = set(all_ids(query))
print("description.clean query:", pretty(query))
print("expected:", len(expected), sorted(expected))
print("actual  :", len(actual), sorted(actual))
assert actual == expected

only_stop = {"match": {"description.clean": "in"}}
with_stop = {"match": {"description": "in"}}
stop_hits = total_hits(es.search(index=INDEX, query=only_stop))
plain_hits = total_hits(es.search(index=INDEX, query=with_stop))
print()
print("query 'in' on description.clean ->", stop_hits, "hits (no tokens left)")
print("query 'in' on description       ->", plain_hits, "hits")
assert stop_hits == 0
assert plain_hits > 0

description.clean query: {
  "match": {
    "description.clean": {
      "query": "Residency in New Panvel",
      "operator": "and"
    }
  }
}
expected: 1 ['PJ001']
actual  : 1 ['PJ001']



query 'in' on description.clean -> 0 hits (no tokens left)
query 'in' on description       -> 50 hits


## Task 3 - Synonyms

Three equivalence groups are installed in a synonym filter. The `synonym`
filter (not `synonym_graph`) is enough because every rule is a set of
single-word equivalents; the filter expands each matching token to the whole
group at index time.

Two things are verified:

1. `_analyze` produces the same expansion set for every member of a group.
2. Searching `description.syn` returns identical project sets for
   apartment/flat, builder/developer and residence/residency, while the plain
   `description` field still misses the synonym-only matches (for example
   `flat` never matches a description that only says "apartment").

In [6]:
print("synonym analyzer definition:")
print(pretty({
    "filter": {
        "business_synonyms_filter": {
            "type": "synonym",
            "synonyms": SYNONYM_GROUPS,
        }
    },
    "analyzer": {
        "synonym_analyzer": {
            "type": "custom",
            "tokenizer": "standard",
            "filter": ["lowercase", "business_synonyms_filter"],
        }
    },
}))

for term in ["flat", "apartment", "builder", "developer", "residence",
             "residency"]:
    print(f"{term:<12} -> {' | '.join(analyze_tokens(term, 'synonym_analyzer'))}")

assert set(analyze_tokens("flat", "synonym_analyzer")) == set(
    analyze_tokens("apartment", "synonym_analyzer")
)
assert set(analyze_tokens("builder", "synonym_analyzer")) == set(
    analyze_tokens("developer", "synonym_analyzer")
)
assert set(analyze_tokens("residence", "synonym_analyzer")) == set(
    analyze_tokens("residency", "synonym_analyzer")
)

synonym analyzer definition:
{
  "filter": {
    "business_synonyms_filter": {
      "type": "synonym",
      "synonyms": [
        "apartment, apartments, flat, flats",
        "builder, builders, developer, developers",
        "residence, residences, residency, residencies"
      ]
    }
  },
  "analyzer": {
    "synonym_analyzer": {
      "type": "custom",
      "tokenizer": "standard",
      "filter": [
        "lowercase",
        "business_synonyms_filter"
      ]
    }
  }
}
flat         -> flat | apartment | apartments | flats
apartment    -> apartment | apartments | flat | flats
builder      -> builder | builders | developer | developers
developer    -> developer | builder | builders | developers


residence    -> residence | residences | residency | residencies
residency    -> residency | residence | residences | residencies


In [7]:
def field_query(field, value):
    return {"match": {field: {"query": value, "operator": "and"}}}


pairs = [
    ("apartment", "flat"),
    ("builder", "developer"),
    ("residence", "residency"),
]
for left, right in pairs:
    left_ids = set(all_ids(field_query("description.syn", left)))
    right_ids = set(all_ids(field_query("description.syn", right)))
    print(f"{left:<10} vs {right:<10} -> {len(left_ids):>2} vs {len(right_ids):>2}")
    assert left_ids == right_ids and left_ids

group_apartment = desc_has("apartment", "apartments", "flat", "flats")
group_builder = desc_has("builder", "builders", "developer", "developers")
group_residence = desc_has(
    "residence", "residences", "residency", "residencies"
)

assert set(all_ids(field_query("description.syn", "flat"))) == group_apartment
assert set(all_ids(field_query("description.syn", "developer"))) == group_builder
assert set(all_ids(field_query("description.syn", "residency"))) == group_residence

baseline = set(all_ids(field_query("description", "flat")))
synonym = set(all_ids(field_query("description.syn", "flat")))
print()
print("plain description, query 'flat':", len(baseline))
print("synonym field,    query 'flat':", len(synonym))
assert 0 < len(baseline) < len(synonym)
print("the synonym field also matches projects described only as apartments")

apartment  vs flat       -> 29 vs 29
builder    vs developer  -> 31 vs 31
residence  vs residency  -> 14 vs 14



plain description, query 'flat': 9
synonym field,    query 'flat': 29
the synonym field also matches projects described only as apartments


## Task 4 - Autocomplete

`name.autocomplete` is indexed with an `edge_ngram` filter (2-15 characters),
so every prefix of every name token is in the index. Queries use
`autocomplete_search`, which does **not** produce n-grams: a typed prefix
such as `gre` stays `gre` and matches the indexed prefix `gre`.

The expected sets are computed from the JSON with the same rule (any name
token starts with the typed prefix), so the assertion is exact. A multi-word
phrase uses `match_phrase`, which keeps the word order: `green val` matches
"Green Valley ..." but not "... Valley Green".

In [8]:
print("analyze 'Green Valley' with autocomplete_index:")
print(" | ".join(analyze_tokens("Green Valley", "autocomplete_index")[:12]), "...")
print("query side with autocomplete_search:")
print(" | ".join(analyze_tokens("Green Val", "autocomplete_search")))

for prefix in ["gre", "sky", "palm", "resi"]:
    query = {"match": {"name.autocomplete": prefix}}
    response = es.search(
        index=INDEX, query=query, size=50, source_includes=["name"]
    )
    expected = name_prefix(prefix)
    print(f"\nautocomplete {prefix!r}: {total_hits(response)} hit(s), "
          f"expected {len(expected)}")
    for hit in hits_of(response)[:3]:
        print(f"  {hit['_id']}  {hit['_source']['name']}")
    assert set(ids_of(response)) == expected

phrase = {"match_phrase": {"name.autocomplete": "green val"}}
expected_phrase = name_token_phrase("green", "val")
response = es.search(
    index=INDEX, query=phrase, size=10, source_includes=["name"]
)
print(f"\nphrase 'green val': {total_hits(response)} hit(s), "
      f"expected {len(expected_phrase)}")
for hit in hits_of(response):
    print(f"  {hit['_id']}  {hit['_source']['name']}")
assert set(ids_of(response)) == expected_phrase

analyze 'Green Valley' with autocomplete_index:
gr | gre | gree | green | va | val | vall | valle | valley ...
query side with autocomplete_search:
green | val

autocomplete 'gre': 9 hit(s), expected 9
  PJ012  River Greens
  PJ014  Green Township
  PJ022  Golden Greens

autocomplete 'sky': 3 hit(s), expected 3
  PJ002  Skyline Heights
  PJ021  Skyline Apartments
  PJ045  Skyline Enclave



autocomplete 'palm': 4 hit(s), expected 4
  PJ006  Royal Palms
  PJ028  Palm Residences
  PJ003  Palm Grove Residences

autocomplete 'resi': 10 hit(s), expected 10
  PJ004  Sunrise Residency
  PJ008  Delta Residency
  PJ009  Horizon Residency

phrase 'green val': 1 hit(s), expected 1
  PJ001  Green Valley Residency


## Task 5 - Partial Search

Edge n-grams only cover prefixes, so a mid-word fragment such as `alley` finds
nothing in `name.autocomplete`. The `name.ngram` field indexes all 3-4
character fragments of every token; the query is analyzed with the same
analyzer, and `operator: and` requires all query fragments to be present.

That finds "Green **Valley** Residency" even though the user typed neither the
start of the name nor the start of a word. A case-insensitive wildcard on
`name.keyword` is used as an independent cross-check of the same set.

N-gram AND matching is not an exact substring test, though: the query `green`
also matches "Sea Breeze Apartments" because `Breeze` supplies the `ree`/`reen`
grams. The notebook surfaces that false positive and shows why wildcard (or a
phrase/position-aware query) is the precise fallback.

In [9]:
print("ngram tokens for 'alley' :", " | ".join(analyze_tokens("alley", "ngram_index")))
print("ngram tokens for 'valley':", " | ".join(analyze_tokens("valley", "ngram_index")))

edge = {"match": {"name.autocomplete": "alley"}}
grams = {"match": {"name.ngram": {"query": "alley", "operator": "and"}}}
wild = {
    "wildcard": {
        "name.keyword": {"value": "*alley*", "case_insensitive": True}
    }
}

for label, query in [
    ("edge autocomplete", edge),
    ("ngram match", grams),
    ("wildcard", wild),
]:
    response = es.search(index=INDEX, query=query, size=50,
                         source_includes=["name"])
    print(f"{label:<18} -> {total_hits(response)} hit(s)")
    for hit in hits_of(response):
        print(f"  {hit['_id']}  {hit['_source']['name']}")

expected = name_contains("alley")
assert set(all_ids(edge)) == set()
assert set(all_ids(grams)) == expected
assert set(all_ids(wild)) == expected
assert expected

for fragment in ["alley", "valley"]:
    ngram_query = {
        "match": {"name.ngram": {"query": fragment, "operator": "and"}}
    }
    wildcard_query = {
        "wildcard": {
            "name.keyword": {
                "value": f"*{fragment}*",
                "case_insensitive": True,
            }
        }
    }
    ngram_ids = set(all_ids(ngram_query))
    wildcard_ids = set(all_ids(wildcard_query))
    print(f"{fragment:<10} ngram={len(ngram_ids):>2} wildcard={len(wildcard_ids):>2}")
    assert ngram_ids == wildcard_ids == name_contains(fragment)

crossed = {"match": {"name.ngram": {"query": "green", "operator": "and"}}}
loose_ids = set(all_ids(crossed))
strict_ids = set(all_ids({
    "wildcard": {
        "name.keyword": {"value": "*green*", "case_insensitive": True}
    }
}))
extra = sorted(loose_ids - strict_ids)
print()
print("cross-token case, query 'green':",
      f"ngram={len(loose_ids)} wildcard={len(strict_ids)} extra={extra}")
assert strict_ids == name_contains("green")
assert strict_ids < loose_ids
assert extra == ["PJ007"]
print("PJ007 'Sea Breeze' supplies the 'ree'/'reen' grams from 'Breeze';")
print("n-gram AND matching does not guarantee the grams are contiguous")
print()
print("edge n-grams cannot find 'alley'; n-grams can, and wildcard agrees")

ngram tokens for 'alley' : all | alle | lle | lley | ley


ngram tokens for 'valley': val | vall | all | alle | lle | lley | ley
edge autocomplete  -> 0 hit(s)
ngram match        -> 1 hit(s)
  PJ001  Green Valley Residency
wildcard           -> 1 hit(s)
  PJ001  Green Valley Residency


alley      ngram= 1 wildcard= 1
valley     ngram= 1 wildcard= 1



cross-token case, query 'green': ngram=10 wildcard=9 extra=['PJ007']
PJ007 'Sea Breeze' supplies the 'ree'/'reen' grams from 'Breeze';
n-gram AND matching does not guarantee the grams are contiguous

edge n-grams cannot find 'alley'; n-grams can, and wildcard agrees


## Task 6 - Case-Insensitive Exact Match

A `normalizer` is the keyword-field equivalent of an analyzer: it applies
`lowercase` and `asciifolding` to the whole value, at index time and at query
time. Exact matching stays exact (no tokenization, no partial matching) but
becomes case-insensitive.

`locality` has the normalizer and keeps `locality.raw` without it as a
control. A `term` query for `NEW PANVEL` matches the normalized field but not
the raw one.

In [10]:
print("normalizer definition:")
print(pretty(INDEX_SETTINGS["analysis"]["normalizer"]))
print("locality mapping:")
print(pretty(MAPPING["properties"]["locality"]))


def term_query(field, value):
    return {"term": {field: value}}


expected = {p["project_id"] for p in DATA if p["locality"] == "New Panvel"}
print("\nprojects in 'New Panvel':", len(expected), sorted(expected))

for value in ["New Panvel", "new panvel", "NEW PANVEL"]:
    normalized = set(all_ids(term_query("locality", value)))
    raw = set(all_ids(term_query("locality.raw", value)))
    print(f"{value:<12} normalized={len(normalized):>2}  raw={len(raw):>2}")
    assert normalized == expected
    if value == "New Panvel":
        assert raw == expected
    else:
        assert raw == set()

for value in ["Navi Mumbai", "navi mumbai", "NAVI MUMBAI"]:
    hit_count = len(all_ids(term_query("city", value)))
    print(f"city {value!r:<16} -> {hit_count} hits")
assert set(all_ids(term_query("city", "NAVI MUMBAI"))) == {
    p["project_id"] for p in DATA if p["city"] == "Navi Mumbai"
}

print()
print("normalized keyword ignores case; raw keyword only matches exact case")

normalizer definition:
{
  "case_insensitive": {
    "type": "custom",
    "filter": [
      "lowercase",
      "asciifolding"
    ]
  }
}
locality mapping:
{
  "type": "keyword",
  "normalizer": "case_insensitive",
  "fields": {
    "raw": {
      "type": "keyword"
    }
  }
}

projects in 'New Panvel': 4 ['PJ001', 'PJ005', 'PJ024', 'PJ030']
New Panvel   normalized= 4  raw= 4


new panvel   normalized= 4  raw= 0


NEW PANVEL   normalized= 4  raw= 0
city 'Navi Mumbai'    -> 19 hits
city 'navi mumbai'    -> 19 hits


city 'NAVI MUMBAI'    -> 19 hits

normalized keyword ignores case; raw keyword only matches exact case


## Summary

| Task | Technique | Evidence |
|------|-----------|----------|
| 1 | Analyze Text | standard 6 tokens, stop 5, keyword 1, english stems |
| 2 | Custom Analyzer | lower_stop drops in/near/the; stop-only query returns 0 |
| 3 | Synonyms | apartment=flat, builder=developer, residence=residency |
| 4 | Autocomplete | edge_ngram 2-15; `gre`, `sky`, `palm`, `resi` match prefixes |
| 5 | Partial Search | `alley` found by n-grams, missed by edge n-grams |
| 6 | Case-Insensitive | normalized term matches any case; raw term is case-sensitive |

### Wrap-up

- Analysis happens twice: at index time with the field analyzer and at query
  time with the search analyzer. They must agree, otherwise matches silently
  disappear.
- Custom stop-word lists replace the default list; include the words you
  still want removed explicitly.
- `synonym` is the index-time filter; use `synonym_graph` when a rule contains
  multiple words and you need query-time multi-word handling.
- `edge_ngram` handles prefixes only; `ngram` handles fragments anywhere in a
  token at the cost of a larger index.
- Normalizers apply to `keyword` fields only and preserve exact (whole-value)
  matching, unlike analyzers on `text` fields.

In [11]:
SUMMARY = [
    ("Task 1", "Analyze Text", "standard/simple/whitespace/stop/keyword/english/lower_stop", "6/6/6/5/1/5/5 tokens"),
    ("Task 2", "Custom Analyzer", "standard + lowercase + stop", "5 tokens, 'in' query -> 0 hits"),
    ("Task 3", "Synonyms", "apartment/flat, builder/developer, residence/residency", "3 equivalent pairs verified"),
    ("Task 4", "Autocomplete", "edge_ngram 2-15 + search analyzer", "gre/sky/palm/resi prefix sets match"),
    ("Task 5", "Partial Search", "ngram 3-4 + wildcard cross-check", "'alley' found by n-grams only"),
    ("Task 6", "Case-Insensitive", "keyword normalizer lowercase+asciifolding", "3 case variants match, raw field does not"),
]

print(f"{'Task':<8} {'Technique':<18} {'Implementation':<60} Evidence")
print("-" * 128)
for task, technique, implementation, evidence in SUMMARY:
    print(f"{task:<8} {technique:<18} {implementation:<60} {evidence}")
print()
print("all six analysis tasks completed with assertions")

Task     Technique          Implementation                                               Evidence
--------------------------------------------------------------------------------------------------------------------------------
Task 1   Analyze Text       standard/simple/whitespace/stop/keyword/english/lower_stop   6/6/6/5/1/5/5 tokens
Task 2   Custom Analyzer    standard + lowercase + stop                                  5 tokens, 'in' query -> 0 hits
Task 3   Synonyms           apartment/flat, builder/developer, residence/residency       3 equivalent pairs verified
Task 4   Autocomplete       edge_ngram 2-15 + search analyzer                            gre/sky/palm/resi prefix sets match
Task 5   Partial Search     ngram 3-4 + wildcard cross-check                             'alley' found by n-grams only
Task 6   Case-Insensitive   keyword normalizer lowercase+asciifolding                    3 case variants match, raw field does not

all six analysis tasks completed with assertions
